# Эксперименты: baseline vs финальная модель

Сравниваем две конфигурации на одинаковом val-сплите Kvasir-SEG:
- **baseline**: vanilla U-Net (`src/models/unet.py`), обучение с нуля, без предобучения (`artifacts/best_unet_baseline.pth`).
- **final**: SwinUNet с энкодером `swin_small_patch4_window7_224`, fine-tune от ImageNet (`artifacts/best_swin_unet.pth`).

Метрики: Accuracy / Precision / Recall / F1 / IoU / Dice.

Условия инференса одинаковы: вход 224×224, нормализация (0.5, 0.5, 0.5), порог бинаризации 0.5.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config import load_config
from src.data.dataset import PolypDataset, build_val_transform, read_split_file
from src.evaluate import evaluate_model
from src.inference import _resolve_device, load_model
from src.metrics import compute_metrics

cfg = load_config(ROOT / 'configs' / 'config.yaml')
device = _resolve_device(None)
print('Device:', device)
print('Доступные модели в конфиге:', list(cfg.model.registry))

In [ ]:
DATA_DIR = cfg.resolve_path(cfg.paths.data_dir)
val_list = read_split_file(DATA_DIR / 'val.txt')
tf = build_val_transform(cfg.inference.image_size,
                         mean=cfg.inference.normalize_mean,
                         std=cfg.inference.normalize_std)

val_ds_named = PolypDataset(val_list, DATA_DIR / 'images', DATA_DIR / 'masks', tf, return_name=True)
val_ds_eval = PolypDataset(val_list, DATA_DIR / 'images', DATA_DIR / 'masks', tf)
val_loader = DataLoader(val_ds_eval, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)
print('Val size:', len(val_ds_eval))

In [ ]:
baseline_model, _, _ = load_model(cfg, device=str(device), model_name='baseline')
baseline_metrics = evaluate_model(baseline_model, val_loader, cfg)
print('Baseline (vanilla U-Net):')
for k, v in baseline_metrics.as_dict().items():
    print(f'  {k:9s}: {v:.4f}')

In [ ]:
final_model, _, _ = load_model(cfg, device=str(device), model_name='final')
final_metrics = evaluate_model(final_model, val_loader, cfg)
print('Final (SwinUNet, Swin-Small):')
for k, v in final_metrics.as_dict().items():
    print(f'  {k:9s}: {v:.4f}')

In [ ]:
b = baseline_metrics.as_dict()
f = final_metrics.as_dict()
print(f'{"metric":<10} {"baseline":>10} {"final":>10} {"delta":>10}')
for k in b:
    print(f'{k:<10} {b[k]:>10.4f} {f[k]:>10.4f} {f[k] - b[k]:>+10.4f}')

In [ ]:
def visualize_side_by_side(idx, threshold=0.5):
    img_t, mask_t, name = val_ds_named[idx]
    img_np = (img_t.cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    gt = mask_t[0].cpu().numpy().astype(np.uint8)

    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(img_np); ax[0].set_title('Изображение'); ax[0].axis('off')
    ax[1].imshow(gt, cmap='gray'); ax[1].set_title('GT маска'); ax[1].axis('off')

    for i, (title, model) in enumerate([('baseline (U-Net)', baseline_model),
                                        ('final (SwinUNet)', final_model)]):
        with torch.no_grad():
            logits = model(img_t.unsqueeze(0).to(device))
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred = (probs > threshold).astype(np.uint8)
        m = compute_metrics(gt, pred)
        ax[2 + i].imshow(img_np)
        ax[2 + i].imshow(pred, cmap='jet', alpha=0.4)
        ax[2 + i].set_title(f'{title}\nIoU={m.iou:.2f} Dice={m.dice:.2f}')
        ax[2 + i].axis('off')
    plt.suptitle(name, fontsize=10)
    plt.tight_layout(); plt.show()

for idx in [9, 25, 50]:
    visualize_side_by_side(idx)

## Выводы

SwinUNet (финальная модель) даёт более высокое качество по IoU/Dice, чем vanilla U-Net (baseline), за счёт предобучения энкодера на ImageNet и глобального контекста Swin Transformer.
Платим за это бОльшим количеством параметров (~57M против ~7.7M) и более долгим инференсом.

Точные числа и обоснование выбора финальной модели - см. [`../report.md`](../report.md).